# DocLens 3/3: prompted-LLM baseline, method comparison, and the query agent (CPU runtime is fine)

1. Gemini zero-shot and 2-shot extraction with a JSON schema (structured output)
2. Compare all methods on the receipts they all share → `results/COMPARISON.md`
3. Load receipts into SQL + embeddings and ask questions through a LangGraph agent (Text-to-SQL + semantic search)

**Needs a Gemini API key** (free at https://aistudio.google.com/apikey). Add it to Colab Secrets as `GOOGLE_API_KEY`.

In [ ]:
REPO_URL = "https://github.com/Harshchoubey55/doclens.git"
import os
os.chdir("/content")
if not os.path.exists("doclens"):
    !git clone -q {REPO_URL} doclens
os.chdir("/content/doclens")
!git pull -q
!pip install -q -r requirements.txt "datasets>=2.19"

from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/doclens"
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
# Model names change over time: list what your key can use and pick a current "flash" model
from google import genai
client = genai.Client()   # keep a reference: the model listing pages lazily through this client
print([m.name for m in client.models.list() if "flash" in m.name][:15])
os.environ["GEMINI_MODEL"] = "gemini-3.8-flash"   # listed models can still be closed to new keys; edit if this one 404s

In [ ]:
import json, time
from datasets import load_dataset
from doclens.cord import DATASET_ID
from doclens.extractors import GeminiExtractor
from doclens.metrics import evaluate
from doclens.predictions import write_jsonl
from doclens.schema import restrict

test = load_dataset(DATASET_ID, split="test")
train = load_dataset(DATASET_ID, split="train")
EVAL_N = 50   # same 50 receipts as the VLM runs; ~6.5 s/request on the free tier
gt = [restrict(json.loads(g)["gt_parse"]) for g in test["ground_truth"]]
write_jsonl("predictions/gt_test.jsonl", [{"doc_id": f"test-{i}", "gt": g} for i, g in enumerate(gt)])

def run(name, extractor):
    rows = []
    for i in range(EVAL_N):
        t0 = time.perf_counter()
        pred = extractor.extract(test[i]["image"])
        rows.append({"doc_id": f"test-{i}", "pred": pred, "latency_s": round(time.perf_counter() - t0, 3),
                     "usage": extractor.last_usage})
    write_jsonl(f"predictions/{name}_test.jsonl", rows)
    s = evaluate([r["pred"] for r in rows], gt[:EVAL_N])
    tok = sum((r["usage"] or {}).get("input_tokens") or 0 for r in rows) / EVAL_N
    print(name, {k: v for k, v in s.items() if k != "per_key_f1"}, f"avg input tokens {tok:.0f}")
    return rows

# free tier: ~10 requests/min -> min_interval_s=6.5
zero = run("llm_zeroshot", GeminiExtractor(min_interval_s=6.5))

In [ ]:
shots = [(train[i]["image"], json.loads(train[i]["ground_truth"])["gt_parse"]) for i in (0, 7)]
few = run("llm_fewshot", GeminiExtractor(few_shot=shots, min_interval_s=6.5))

## Compare all methods
Copies in the LayoutLMv3 and QLoRA predictions that notebooks 1 and 2 saved to Drive.

In [ ]:
!cp -n {DRIVE}/predictions/*.jsonl predictions/ 2>/dev/null; ls predictions
!python scripts/compare.py predictions results
!cp -r results predictions {DRIVE}/

## Query agent
All 800 train receipts (ground truth) + test receipts as extracted by the best model go into SQLite.
Store names and dates are simulated (CORD has none) so time/vendor questions are possible.

In [ ]:
from doclens.predictions import read_jsonl
from doclens.store import Store

if os.path.exists("doclens.db"):
    os.remove("doclens.db")
store = Store("sqlite:///doclens.db")
for i, g in enumerate(train["ground_truth"]):
    store.add_receipt(f"train-{i}", json.loads(g)["gt_parse"], "ground_truth")
best = json.load(open("results/comparison.json"))["methods"]
best_name = max(best, key=lambda m: best[m]["field_f1"])
for doc_id, r in read_jsonl(f"predictions/{best_name}_test.jsonl").items():
    store.add_receipt(doc_id, r["pred"], best_name)
print("best extractor:", best_name, "| items indexed:", store.build_index())
store.read_only_query("SELECT source, COUNT(*) AS n, ROUND(AVG(total)) AS avg_total FROM receipts GROUP BY source")

In [ ]:
from doclens.agent import ask, build_agent

agent = build_agent(store)
questions = [
    "How many receipts are there, and what is the average bill?",
    "Which store has the highest total revenue, and how much?",
    "Find items similar to 'iced tea' and tell me how many units of them were sold in total.",
    "What about coffee drinks?",                         # follow-up: relies on conversation memory
    "What share of receipts were paid by credit card?",
    "Delete all receipts from Kopi Senja.",             # should be refused (read-only tools)
]
for q in questions:
    out = ask(agent, q, thread_id="demo")
    print(f"\nQ: {q}\nA: {out['answer']}")
    for c in out["tool_calls"]:
        print("   ->", c["tool"], json.dumps(c["args"])[:200])